# Complexity-03h - L'exposant du produit matriciel est 9/4

**Serie Complexity** - accretion 03h - preprint OpenAI du 24 novembre 2026

**Navigation** : [Index](README.md) | [Carnet precedent - 03f, la chaine des reductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb) | [Carnet suivant - 04, algorithmes en ligne](Complexity-04-OnlineAlgorithms-Python.ipynb)

## Provenance

| Element | Valeur |
|---|---|
| Texte | *The matrix multiplication exponent is nine fourths*, OpenAI, 24/11/2026, **111 pages** |
| Depot | `openai/math` (Apache-2.0), `preprints/The-matrix-multiplication-exponent-is-nine-fourths-November-24-2026/paper.pdf`, **1 075 142 octets**, sha256 `ddde9ab7755596c6...` |
| Copie archivee | `G:\Mon Drive\MyIA\IA\Bibliographie IA\Algorithms\2026 - OpenAI - The matrix multiplication exponent is nine fourths.pdf` |
| Papier compagnon | *Euclidean geometry in the asymptotic spectrum of tensors*, meme depot, meme jour (reference [42] du papier principal), 861 799 octets, archive au meme endroit |
| Statut | preprint **auto-publie**, **non relu par les pairs** ; aucune verification independante rapportee a la date d'ecriture de ce carnet |

**Regle de lecture de ce carnet.** Tout enonce numerote (Theoreme 1.1, Corollaires 1.2 et 1.3, Lemme 2.4, Theoremes 2.1 et 2.2) est **RAPPORTE** du preprint : ce carnet ne re-prouve pas la borne inferieure. Ce qu'il fait, il le fait pour de bon : il calcule les quantites que ces enonces manipulent, il verifie l'identite de Strassen a la machine, et il dit exactement ou s'arrete ce qui est demontre ici.

## 1. Ce qu'est l'exposant, et ce que la serie savait deja

Multiplier deux matrices carrees de cote $n$ demande $n^3$ produits scalaires si l'on calcule chaque coefficient separement. L'exposant du produit matriciel, note $\omega$, est la plus petite puissance $s$ telle que le produit se calcule en $O(n^{s+\varepsilon})$ operations pour tout $\varepsilon > 0$.

L'identite de Strassen (1969) montre que partager du travail entre les coefficients fait gagner une puissance de $n$ : sept produits suffisent pour des blocs $2 \times 2$, et la recursion donne $O(n^{\log_2 7})$. La conjecture $\omega = 2$ demandait si l'on pouvait descendre jusqu'a la taille de la **sortie** ($n^2$ coefficients).

Le preprint du 24/11/2026 repond : l'exposant depasse l'exposant de la taille de sortie d'**exactement un quart**.

Les trois premiers carnets de la serie posaient le decor : 03e explique les barrieres sur les exposants, 07c constate qu'elles ont casse. 03h est le carnet du **resultat qui les clot**.

In [1]:
import math

# Trois reperes sur l'axe des exposants.
strassen = math.log2(7)
exact = 9 / 4

print(f"produit naif            : n^3          -> exposant 3")
print(f"Strassen 1969           : n^{strassen:.4f}     -> exposant {strassen:.4f}")
print(f"borne du 24/11/2026     : n^(9/4)      -> exposant {exact:.4f}")
print(f"taille de la sortie     : n^2          -> exposant 2")
print()
for nom, w in (("Strassen", strassen), ("9/4 (24/11/2026)", exact)):
    print(f"{nom:>18} : gain d'exposant sur le naif = {3 - w:+.4f}")

produit naif            : n^3          -> exposant 3
Strassen 1969           : n^2.8074     -> exposant 2.8074
borne du 24/11/2026     : n^(9/4)      -> exposant 2.2500
taille de la sortie     : n^2          -> exposant 2

          Strassen : gain d'exposant sur le naif = +0.1926
  9/4 (24/11/2026) : gain d'exposant sur le naif = +0.7500


### Lecture du resultat

Les trois valeurs utiles : $3$ (calcul coefficient par coefficient), $\log_2 7 \approx 2{,}807$ (Strassen), et $9/4 = 2{,}25$ (le preprint). La conjecture $\omega = 2$ aurait place l'exposant sur la taille de la sortie ; le resultat le place un quart au-dessus, et **exactement** au-dessus : il ne s'agit plus d'une suite d'ameliorations mais d'une valeur.

Un detail de vocabulaire compte pour toute la suite : $\omega$ est un **infimum** et la borne est asymptotique. Le Theoreme 1.1 ne fournit aucune taille de matrice finie ou l'algorithme serait competitif, et le preprint le dit explicitement.

## 2. Le modele : circuits arithmetiques formels

Le cadre du preprint n'est pas celui des algorithmes bilineaires seuls, mais celui des **circuits arithmetiques formels** (verbatim du §1, RAPPORTE) :

- les entrees sont des **variables independantes** ;
- les constantes appartiennent au corps $F$ ;
- chaque porte additionne, soustrait, multiplie, ou **divise par une fonction rationnelle non nulle** ;
- chaque sortie doit coincider avec l'entree correspondante du produit **comme identite de fonctions rationnelles**.

La derniere clause est celle qui porte tout le poids, et elle est ce qui rend l'enonce valable **sur les corps finis** : sur $\mathbb{F}_2$, deux fonctions peuvent coincider sur toutes les entrees sans etre la meme identite. Exiger l'identite de fonctions rationnelles, c'est exiger une preuve symbolique, valable sur toute extension.

Le cout compte les portes **non scalaires** seulement : additions et soustractions sur des scalaires sont gratuites. C'est le regime ou une borne inferieure est enonçable.

In [2]:
from decimal import Decimal, getcontext

getcontext().prec = 40


def portes_minimales(n: int) -> int:
    """Borne du 24/11 : ceil(n^(9/4) / 2) portes non scalaires, pour tout n >= 2.

    Calcul en decimal haute precision : n^(9/4) = (n^9)^(1/4), et la racine
    quatrieme est prise par deux racines carrees successives.
    """
    quatrieme = Decimal(n ** 9).sqrt().sqrt()
    return int((quatrieme / 2).to_integral_value(rounding="ROUND_CEILING"))


# Table : la borne face a la taille de la sortie (n^2) et face au calcul naif (n^3).
# La ligne n = 17 est la premiere ou la borne depasse n^2 ; elle est affichee avec sa voisine.
print(f"{'n':>4} {'n^2 (sortie)':>13} {'borne n^(9/4)/2':>17} {'n^3 (naif)':>12} {'n^3 / borne':>13}")
for n in (2, 3, 4, 5, 8, 10, 16, 17, 20, 50, 100):
    borne = portes_minimales(n)
    print(f"{n:>4} {n**2:>13} {borne:>17} {n**3:>12} {n**3 / borne:>12.1f}x")

# Deux proprietes lues sur toute la plage n = 2 a 100, pas seulement sur les lignes affichees
depasse_sortie = [n for n in range(2, 101) if portes_minimales(n) > n ** 2]
sous_le_naif = all(portes_minimales(n) < n ** 3 for n in range(2, 101))
print()
print(f"plus petit n ou la borne depasse n^2 : {depasse_sortie[0]}")
print(f"borne toujours strictement sous n^3 sur 2..100 : {sous_le_naif}")

   n  n^2 (sortie)   borne n^(9/4)/2   n^3 (naif)   n^3 / borne
   2             4                 3            8          2.7x
   3             9                 6           27          4.5x
   4            16                12           64          5.3x
   5            25                19          125          6.6x
   8            64                54          512          9.5x
  10           100                89         1000         11.2x
  16           256               256         4096         16.0x
  17           289               294         4913         16.7x
  20           400               423         8000         18.9x
  50          2500              3324       125000         37.6x
 100         10000             15812      1000000         63.2x

plus petit n ou la borne depasse n^2 : 17
borne toujours strictement sous n^3 sur 2..100 : True


### Lecture du resultat

La colonne du milieu est une **borne inferieure** : aucun circuit, aussi ingenieux soit-il, ne descend en dessous. Face au calcul naif, la marge est nette et grandit avec $n$ - de $2{,}7$ fois a $n = 2$ a $63{,}2$ fois a $n = 100$ : c'est ce que veut dire « gagner une puissance ».

Face a la taille de la sortie ($n^2$), en revanche, la borne ne se place pas au-dessus. Le tableau le montre a ses deux bouts, et le balayage de $n = 2$ a $100$ imprime le point de bascule :

- a $n = 2$, la borne vaut $3$ portes pour $4$ coefficients de sortie : elle est **en dessous** de $n^2$ ;
- a $n = 16$, les deux valent exactement $256$ : la borne **egale** $n^2$ sans le depasser ;
- c'est a partir de $n = 17$ qu'elle depasse $n^2$ ($294$ portes contre $289$).

La raison est arithmetique et sans mystere : $n^{9/4}/2 > n^2$ equivaut a $n^{1/4} > 2$, donc a $n > 16$. Ce n'est pas un defaut du resultat, c'est la nature d'une borne d'exposant : elle est asymptotique, et le facteur $2$ du Lemme 2.4 (section suivante) la laisse sous $n^2$ aux petites tailles. Le progres qu'elle porte se lit dans l'ecart au calcul naif, pas dans une comparaison a la taille de la sortie.

Attention a ne pas lire ce tableau a l'envers : a $n = 100$, la borne est $15\,812$ portes alors que l'algorithme naif en demande $1\,000\,000$. Cela n'affirme **pas** qu'un circuit de $15\,812$ portes existe : c'est le role de la borne **superieure**, $O(n^{9/4+\varepsilon})$, qui est asymptotique et n'est pas constructive a taille finie.

## 3. D'ou vient le facteur 2 (et pourquoi ce n'est pas une constante arbitraire)

La borne porte sur les portes non scalaires, mais la theorie historique compte des **produits** (le rang du tenseur). Le passage de l'un a l'autre est un enonce nomme, le **Lemme 2.4** (bilinearisation, RAPPORTE) :

> Si un circuit pour des matrices $s \times s$ utilise $q$ portes non scalaires, alors $\operatorname{rank}_F(T_s) \le 2q$, d'ou $q \ge \lceil n^{9/4}/2 \rceil$.

Une porte de multiplication ou de division introduit **au plus deux** produits de formes lineaires nouvelles. C'est le seul endroit ou le facteur $2$ entre dans l'enonce : il vient de ce compte-la, pas d'une convention de constante.

Le cas $s = 2$ est verifiable a la main : le rang de $T_2$ est $7$ (Strassen) contre $8$ produits pour le calcul naif. Verifions-le a la machine.

In [3]:
import random

random.seed(0)


def produit_naif(A, B):
    """Produit matriciel par la definition : n**3 multiplications scalaires."""
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]


def strassen_2x2(A, B):
    """Identite de Strassen : un 2x2 en sept produits scalaires au lieu de huit.

    Seules les multiplications coutent dans le modele algebrique ; les
    additions et soustractions sont gratuites. Les sept produits m1..m7 et
    les combinaisons additives finales sont ceux de l'article de 1969.
    """
    (a, b), (c, d) = A
    (e, f), (g, h) = B
    m1 = (a + d) * (e + h)
    m2 = (c + d) * e
    m3 = a * (f - h)
    m4 = d * (g - e)
    m5 = (a + b) * h
    m6 = (c - a) * (e + f)
    m7 = (b - d) * (g + h)
    # Chacun des quatre coefficients du resultat est une somme de trois des m_i.
    return [[m1 + m4 - m5 + m7, m3 + m5],
            [m2 + m4, m1 - m2 + m3 + m6]]


# Verification a la machine : sur 200 couples d'entiers, les deux calculs
# doivent coincider. Les entiers evitent tout flottement, donc l'egalite
# testee est exacte et non numerique.
identiques = True
for _ in range(200):
    A = [[random.randint(-9, 9) for _ in range(2)] for _ in range(2)]
    B = [[random.randint(-9, 9) for _ in range(2)] for _ in range(2)]
    identiques = identiques and strassen_2x2(A, B) == produit_naif(A, B)

print("identite de Strassen verifiee sur 200 couples entiers :", identiques)
print("produits scalaires : 7 (Strassen) contre 8 (naif) -> rapport", 7 / 8)
print("exposant de la recursion 2x2 : log2(7) =", round(math.log2(7), 6))


identite de Strassen verifiee sur 200 couples entiers : True
produits scalaires : 7 (Strassen) contre 8 (naif) -> rapport 0.875
exposant de la recursion 2x2 : log2(7) = 2.807355


### Lecture du resultat

L'egalite est verifiee sur des entiers, donc sans erreur d'arrondi : les deux expressions sont polynomiales a coefficients entiers, et 200 couples suffisent a rendre une erreur de decomposition tres improbable sur des valeurs aleatoires. Sept produits au lieu de huit, c'est $7/8$ par niveau de recursion, et $7/8$ eleve au nombre de niveaux donne l'exposant $\log_2 7$.

C'est la borne **superieure** dans sa forme la plus simple. Le preprint ne l'utilise pas : il prend pour acquises les bornes superieures de 2026 ($9/4$ sur $\mathbb{C}$ par OpenAI le 02/10, puis sur tout corps) et demontre la borne **inferieure** appariée.

## 4. L'architecture de la preuve, en une page

L'argument se lit en trois temps (RAPPORTE du preprint, §1.2 et §2).

**1. Une contradiction, vue de haut.** Supposons que l'on gagne une puissance fixe sous $9/4$. Un algorithme bilineaire temoin, specialise a un corps fini, permet alors de **compresser** les puissances tensorielles d'un reseau (Theoreme 2.1) au point de violer une borne inferieure de **volume** pour le test de **distinctness de listes** (Theoreme 2.2, constante $\sigma > 0$). Ce test -「 les $2^w$ mots d'une liste sont-ils tous distincts ? 」- n'a rien a voir avec le produit matriciel : la borne inferieure est **importee d'un autre domaine**, et c'est la que reside l'originalite.

**2. La contradiction courte.** D'abord les corps finis (Proposition 2.3 : sur tout corps fini $k$, $\omega_k \ge 9/4$), puis une **specialisation finie** (Lemme 2.5 : si $\omega_F < 9/4$ sur $F$, alors $\omega_k < 9/4$ sur un corps fini $k$) qui ramene le cas general au cas fini.

**3. La machinerie.** Le reste - sections 3 a 27 - construit la compression : prix de tenseurs et distances de fil (§3, §5-6), tenseurs spheriques fermioniques et leur prix (§13-14), expansion d'incidence (§15), routage de permutations a largeur bornee (§16-18), concentration de l'aritie d'un reseau et insertion sur surfaces cousues (§19-25), enfin compression trivalente et exposant (§27).

Le carnet n'entre pas dans les spheres fermioniques : il en donne la carte et le role. La geometrie vient du **papier compagnon** [42], qui n'est pas un supplement mais le cadre sans lequel la borne ne s'enonce pas.

## 5. La cloture : ce que 9/4 change, et ce que ce carnet ne prouve pas

Le Theoreme 1.1 (RAPPORTE) : sur tout corps $F$, l'exposant du produit de matrices carrees vaut $\omega_F = 9/4$. Comme $9/4 > 2$, la conjecture $\omega = 2$ est **refutee**.

Le Corollaire 1.2 (RAPPORTE) donne la forme par taille : pour tout $n \ge 2$ et tout corps, la **border rank geometrique** de $T_n$ est au moins $n^{9/4}$, et sa **rank asymptotique** sur $F$ vaut $n^{9/4}$. En particulier, un algorithme bilineaire exact a un cote $n$ utilise au moins $\lceil n^{9/4} \rceil$ produits.

Le Corollaire 1.3 (RAPPORTE) traite le cas rectangulaire : pour $T_{a,b,c}$ (une matrice $a \times b$ par une $b \times c$), la border rank geometrique et la rank asymptotique valent au moins $\max\{ab, bc, ca, (abc)^{3/4}\}$, et l'exposant rectangulaire verifie $1/2 \le \alpha_F \le 2/3$.

**Ce que ce carnet ne fait pas.** Il ne re-prouve aucun de ces enonces ; il ne verifie pas la formalisation Lean associee ; il ne remplace pas une relecture par les pairs, qui n'a pas eu lieu. Une distillation qui laisserait croire le contraire serait un faux temoignage - et c'est precisement ce que la serie Complexity doit a ses lecteurs.

In [4]:
# Corollaire 1.3 : le cas rectangulaire, et la plage de alpha.


def quatre_termes(n: int, beta: float):
    """Les quatre quantites de max{ab, bc, ca, (abc)^(3/4)} pour a = n, b = n^beta, c = n."""
    a, c = n, n
    b = max(1, int(n ** beta))
    return {"ab": a * b, "bc": b * c, "ca": c * a, "(abc)^(3/4)": (a * b * c) ** 0.75}


n = 4096
print(f"cas rectangulaire, a = c = n = {n}, b = floor(n^beta)")
print(f"{'beta':>6} {'b':>7} {'ab':>12} {'bc':>12} {'ca':>12} {'(abc)^(3/4)':>15} {'max':>15}")
for beta in (0.5, 0.6, 2 / 3, 1.0):
    termes = quatre_termes(n, beta)
    maximum = max(termes.values())
    gagnant = max(termes, key=termes.get)
    print(f"{beta:>6.3f} {max(1, int(n ** beta)):>7} {termes['ab']:>12} {termes['bc']:>12} "
          f"{termes['ca']:>12} {termes['(abc)^(3/4)']:>15.0f} {gagnant:>15}")

print()
print("pour a = b = c = n : (abc)^(3/4) = n^(9/4), et il domine n^2 des que n > 1")
for n_carre in (2, 4, 16, 1024):
    print(f"  n = {n_carre:>5} : n^2 = {n_carre**2:>8}, n^(9/4) = {(n_carre**3) ** 0.75:>12.1f}")

cas rectangulaire, a = c = n = 4096, b = floor(n^beta)
  beta       b           ab           bc           ca     (abc)^(3/4)             max
 0.500      64       262144       262144     16777216         5931642              ca
 0.600     147       602112       602112     16777216        11066949              ca
 0.667     255      1044480      1044480     16777216        16728040              ca
 1.000    4096     16777216     16777216     16777216       134217728     (abc)^(3/4)

pour a = b = c = n : (abc)^(3/4) = n^(9/4), et il domine n^2 des que n > 1
  n =     2 : n^2 =        4, n^(9/4) =          4.8
  n =     4 : n^2 =       16, n^(9/4) =         22.6
  n =    16 : n^2 =      256, n^(9/4) =        512.0
  n =  1024 : n^2 =  1048576, n^(9/4) =    5931641.6


### Lecture du resultat

Pour $a = b = c = n$, le terme $(abc)^{3/4}$ vaut $n^{9/4}$ et domine $n^2$ : le corollaire rectangulaire **contient** le cas carre, ce qui est le sens de la puissance $3/4$ - trois copies du tenseur rectangulaire, roles des facteurs permutes, se multiplient en le tenseur carre $T_{abc}$.

La plage $1/2 \le \alpha_F \le 2/3$ se lit dans le tableau : a $\beta = 1$, on est ramene au cas carre ; a $\beta = 1/2$, on est a la frontiere connue par les algorithmes a dimension interne $\lfloor\sqrt{n}\rfloor$. $\alpha_F$ mesure jusqu'ou un produit rectangulaire peut s'elargir tout en gardant le cout d'exposant deux - et 9/4 en fixe le plafond.

## Exercices

Trois exercices qui ne demandent aucun outil supplementaire : tout se calcule avec la bibliotheque standard Python. Les cellules sont des **stubs** : elles s'executent telles quelles et n'affichent que la consigne.

In [5]:
# Exercice 1 - la borne de portes a grande taille, en arithmetique exacte
#
# Enonce : produire un tableau (n, borne, n^3 / borne) pour n = 100, 1 000, 10 000,
# ou borne = ceil(n^(9/4) / 2). Contrainte : pas de flottant pour la borne, elle doit
# rester exacte - la fonction portes_minimales de la section 2 est reutilisable.
#
# Attendu : trois lignes, et une phrase sur ce que le rapport n^3 / borne mesure.
# TODO etudiant
resultat = None  # TODO etudiant : tableau des trois tailles
print("Exercice a completer : (n, ceil(n^(9/4)/2), n^3/borne) pour n = 100, 1000, 10000")

Exercice a completer : (n, ceil(n^(9/4)/2), n^3/borne) pour n = 100, 1000, 10000


In [6]:
# Exercice 2 - compter les portes d'une recursion de Strassen
#
# Enonce : ecrire portes_strassen(k) = nombre de produits scalaires pour multiplier
# deux matrices n x n avec n = 2^k par recursion de Strassen jusqu'a des blocs 2 x 2.
# Verifier portes_strassen(1) = 7, puis comparer portes_strassen(k) a n^3 pour k = 1..10.
#
# Indice : chaque niveau divise le cote par deux et multiplie le nombre d'appels par 7.
# TODO etudiant
resultat = None  # TODO etudiant : liste des couples (k, portes_strassen(k), n^3)
print("Exercice a completer : portes_strassen(k) contre n^3, pour k = 1..10")

Exercice a completer : portes_strassen(k) contre n^3, pour k = 1..10


In [7]:
# Exercice 3 - retrouver le corollaire 1.3 et la plage de alpha
#
# Enonce : (a) montrer par le calcul que pour a = b = c = n,
# max{ab, bc, ca, (abc)^(3/4)} = n^(9/4) ;
# (b) pour beta dans {1/2, 2/3, 1}, dire lequel des quatre termes atteint le maximum
# et relier le resultat aux bornes 1/2 <= alpha_F <= 2/3 du Corollaire 1.3.
#
# TODO etudiant
resultat = None  # TODO etudiant : (a) egalite, (b) gagnant par beta
print("Exercice a completer : quatre termes du corollaire 1.3, puis lecture de alpha")

Exercice a completer : quatre termes du corollaire 1.3, puis lecture de alpha


## Conclusion

Trois acquis a garder de ce carnet.

1. **Une valeur, pas une suite.** L'exposant du produit matriciel vaut $9/4$ sur tout corps (RAPPORTE) : les bornes superieures de 2026 sont appariees a une borne inferieure du meme jour. La conjecture $\omega = 2$ est refutee.
2. **Un modele explicite.** La borne porte sur les portes non scalaires d'un circuit arithmetique formel, avec egalite comme identite de fonctions rationnelles - la clause qui la rend valable sur les corps finis. Le facteur $2$ vient du Lemme 2.4, pas d'une convention.
3. **Une borne inferieure importee.** L'argument ne raffine pas les methodes de produit matriciel : il specialise un algorithme hypothetique, compresse un reseau de tenseurs, et viole une borne de volume empruntee au test de distinctness de listes.

## References

- **Source principale** : OpenAI, *The matrix multiplication exponent is nine fourths*, 24/11/2026, 111 p. - copie archivee : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Algorithms\2026 - OpenAI - The matrix multiplication exponent is nine fourths.pdf` (sha256 `ddde9ab7755596c6...`).
- **Papier compagnon [42]** : OpenAI, *Euclidean geometry in the asymptotic spectrum of tensors*, 24/11/2026 - archive au meme endroit (`...\2026 - OpenAI - Euclidean geometry in the asymptotic spectrum of tensors.pdf`).
- **Borne superieure, version du 02/10/2026** : OpenAI, *An upper bound of 9/4 for the matrix multiplication exponent*, 13 p. - archive au meme endroit.
- **Precedents de la serie** : `Complexity-03e` (barrieres sur les exposants) et `Complexity-07c` (le bilan Mathocalypse).
- **Depot** : `openai/math`, Apache-2.0 ; dossier `preprints/The-matrix-multiplication-exponent-is-nine-fourths-November-24-2026/`, qui contient aussi une formalisation Lean (`lean/OAI/LinearAlgebra/SpectralBooks/ExactExponent.lean`).

> Statut de la source : preprint auto-publie, non relu par les pairs, sans verification independante rapportee a la date d'ecriture.

***

*Carnet 03h de la serie Complexity. Verifie : les PDF sont archives et leur empreinte sha256 a ete controlee a l'arrivee ; l'identite de Strassen est verifiee a la machine dans ce carnet. Rapporte : tous les enonces numerotes du preprint.*